# Wetland Loss to Agriculture in a Prairie Pothole Watershed, 2000–2020

**Question:** How much wetland in the Pipestem watershed (HUC8, North Dakota) was converted to cropland between 2000 and 2020?

**Approach in one sentence:** build cloud-free spring Landsat composites for 2000, 2010 and 2020, train a separate
random forest for each snapshot using that period's Cropland Data Layer and the National Wetlands Inventory as labels,
classify each snapshot, and track pixels that went from wetland to cropland.

**Why 2000–2020 and not 1990?** A first version trained one model on 2020 and applied it to 1990, 2000 and 2010.
Sanity checks showed that model did not carry over to older imagery: it mapped implausibly large wetland areas in the
earlier years, including field-shaped "wetlands" during the 1989–1991 drought. Testing variants (with and without a
Landsat 8 → Landsat 7 adjustment, and with indices only) did not fix it (`scripts/diagnose_transfer.py`). The Cropland
Data Layer covers this watershed every year from 1997, so restricting the study to 2000–2020 lets every snapshot have
**its own training labels and its own model**.

Every number reported in this project comes from a cell in this notebook.

## Step 0 — Connect to Earth Engine

Earth Engine runs the heavy processing on Google's servers; this notebook just sends it instructions.
To use it you need a (free, noncommercial) Google Cloud project registered for Earth Engine.
`ee.Initialize(project=...)` tells Earth Engine which project to bill the (free) usage to.

In [1]:
import ee
import geemap

# Your Google Cloud project registered for noncommercial Earth Engine use
PROJECT_ID = "agriculture-wetland-loss"

# Authenticate() only opens a browser the first time; after that it reuses saved credentials.
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print("Earth Engine initialized:", ee.String("ok").getInfo())

Earth Engine initialized: ok


## Step 1a — Load the watershed boundary

The USGS Watershed Boundary Dataset (WBD) divides the U.S. into nested hydrologic units.
A **HUC8** is a sub-basin, typically a few thousand km². We look up Pipestem by name and print its
code and area so we can confirm we have the right polygon before doing anything else.

In [2]:
huc8 = ee.FeatureCollection("USGS/WBD/2017/HUC08")
watershed = huc8.filter(ee.Filter.eq("name", "Pipestem"))

# Check what we actually got (there should be exactly one feature, in ND)
info = watershed.first().toDictionary().getInfo()
print("Features matched:", watershed.size().getInfo())
print({k: info[k] for k in sorted(info)})

aoi = watershed.geometry()  # the area of interest used everywhere below
print("Area computed from geometry (km²):", round(aoi.area(maxError=1).divide(1e6).getInfo(), 1))

Features matched: 1
{'areaacres': '685100.43', 'areasqkm': '2772.51', 'gnis_id': '', 'huc8': '10160002', 'loaddate': '20120611075457', 'metasource': '', 'name': 'Pipestem', 'shape_area': '0.329154073822313', 'shape_leng': '4.36477122786687', 'sourcedata': '', 'sourcefeat': '', 'sourceorig': '', 'states': 'ND', 'tnmid': '{C1EC565A-3D79-4C86-9D4E-B1DE84BABDB3}'}


Area computed from geometry (km²): 2764.9


## Step 1b — Cloud masking, scaling, and a common band naming

**Landsat Collection 2 Level-2** gives *surface reflectance*: the fraction of sunlight reflected by the ground,
with atmospheric effects removed. A few housekeeping steps are needed for every image:

1. **Cloud mask.** Each scene has a `QA_PIXEL` band where individual bits flag fill, cloud, dilated cloud,
   cloud shadow and snow. We drop any pixel with one of those flags, plus saturated pixels (`QA_RADSAT`).
2. **Scale factors.** Reflectance is stored as integers; `value × 0.0000275 − 0.2` converts to 0–1 reflectance.
3. **Common band names.** Landsat 5/7 and Landsat 8 number their bands differently
   (e.g. red is `SR_B3` on Landsat 5/7 but `SR_B4` on Landsat 8), so we rename to `blue, green, red, nir, swir1, swir2`.

Note: Landsat 8 sees slightly different wavelengths than Landsat 5/7. We use its surface reflectance as-is
(no cross-sensor adjustment), which keeps the method simple; the possible effect is noted under limitations in the README.

In [3]:
BANDS = ["blue", "green", "red", "nir", "swir1", "swir2"]
TM_ETM_BANDS = ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"]  # Landsat 5 and 7
OLI_BANDS = ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]     # Landsat 8


def mask_clouds(img):
    """Keep only clear pixels using the QA_PIXEL bit flags."""
    qa = img.select("QA_PIXEL")
    # bit 0 fill, bit 1 dilated cloud, bit 3 cloud, bit 4 cloud shadow, bit 5 snow
    bad = (1 << 0) | (1 << 1) | (1 << 3) | (1 << 4) | (1 << 5)
    clear = qa.bitwiseAnd(bad).eq(0)
    not_saturated = img.select("QA_RADSAT").eq(0)
    return img.updateMask(clear).updateMask(not_saturated)


def scale(img, band_names):
    """Convert stored integers to surface reflectance and rename to common names."""
    return img.select(band_names, BANDS).multiply(0.0000275).add(-0.2)


def prep_tm_etm(img):
    return scale(mask_clouds(img), TM_ETM_BANDS).copyProperties(img, ["system:time_start"])


def prep_oli(img):
    return scale(mask_clouds(img), OLI_BANDS).copyProperties(img, ["system:time_start"])

## Step 1c — Build the three spring composites

For each snapshot year we take **every clear Landsat pixel from April–June over a 3-year window** and keep the
**median** value per pixel. The median ignores leftover clouds or odd values, and the 3-year window
smooths out a single unusually wet or dry spring.

| Snapshot | Window | Sensors |
|---|---|---|
| 2000 | 1999–2001 | Landsat 5 + 7 |
| 2010 | 2009–2011 | Landsat 5 + 7 (Landsat 7 has striping gaps after 2003; other scenes fill them) |
| 2020 | 2019–2021 | Landsat 8 |

We print how many scenes went in and how many clear observations each pixel had, so we know the composites are well supported.

In [4]:
COLLECTIONS = {
    "L5": ("LANDSAT/LT05/C02/T1_L2", prep_tm_etm),
    "L7": ("LANDSAT/LE07/C02/T1_L2", prep_tm_etm),
    "L8": ("LANDSAT/LC08/C02/T1_L2", prep_oli),
}

SNAPSHOTS = {
    2000: (1999, 2001, ["L5", "L7"]),
    2010: (2009, 2011, ["L5", "L7"]),
    2020: (2019, 2021, ["L8"]),
}


def spring_collection(first_year, last_year, sensors):
    """All cloud-masked April-June scenes over the AOI for the window, merged across sensors."""
    merged = ee.ImageCollection([])
    for s in sensors:
        collection_id, prep = COLLECTIONS[s]
        col = (
            ee.ImageCollection(collection_id)
            .filterBounds(aoi)
            .filterDate(f"{first_year}-01-01", f"{last_year + 1}-01-01")
            .filter(ee.Filter.calendarRange(4, 6, "month"))
            .map(prep)
        )
        merged = merged.merge(col)
    return merged


composites = {}
for year, (y0, y1, sensors) in SNAPSHOTS.items():
    col = spring_collection(y0, y1, sensors)
    composites[year] = col.median().clip(aoi)

    # Clear observations per pixel = how many scenes had a usable value there
    n_clear = col.select("red").count().clip(aoi)
    stats = n_clear.reduceRegion(
        reducer=ee.Reducer.min().combine(ee.Reducer.mean(), sharedInputs=True),
        geometry=aoi, scale=30, maxPixels=1e10,
    ).getInfo()
    print(f"{year} ({y0}-{y1}, {'+'.join(sensors)}): {col.size().getInfo()} scenes | "
          f"clear obs per pixel: min {stats['red_min']}, mean {stats['red_mean']:.1f}")

2000 (1999-2001, L5+L7): 43 scenes | clear obs per pixel: min 1, mean 14.5


2010 (2009-2011, L5+L7): 57 scenes | clear obs per pixel: min 2, mean 23.6


2020 (2019-2021, L8): 45 scenes | clear obs per pixel: min 4, mean 13.2


## Step 1d — Look at the composites

Two views per year:
- **True color** (red, green, blue): what your eye would see.
- **False color** (SWIR1, NIR, red): vegetation shows green, open water and wet soils show dark blue/black, and bare
  fields look pink/brown. This is the easiest way to spot prairie potholes.

Toggle layers in the top-right control to compare years.

In [5]:
m = geemap.Map(basemap="Esri.WorldImagery")
m.centerObject(aoi, 10)

true_color = {"bands": ["red", "green", "blue"], "min": 0, "max": 0.2}
false_color = {"bands": ["swir1", "nir", "red"], "min": 0, "max": 0.35}

for year, img in composites.items():
    m.addLayer(img, true_color, f"{year} true color", shown=False)
    m.addLayer(img, false_color, f"{year} false color", shown=(year in (2000, 2020)))

m.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": "yellow"}, "Pipestem boundary")
m

## Step 2a — Add spectral indices

Raw bands are useful, but a few **indices** (simple band math) make the three classes much easier to separate:

| Index | Formula | What it highlights |
|---|---|---|
| **NDVI** (Normalized Difference Vegetation Index) | (NIR − Red) / (NIR + Red) | Green, growing vegetation is high; water is negative; bare spring fields are low. |
| **MNDWI** (Modified Normalized Difference Water Index, Xu 2006) | (Green − SWIR1) / (Green + SWIR1) | Open water is positive; land is mostly negative. Uses SWIR1 because water absorbs it strongly. |
| **Tasseled Cap Wetness** (Crist 1985) | weighted sum of all six bands | Soil and plant moisture. Higher = wetter. Helps with shallow, vegetated wetlands that aren't open water. |

Spring timing helps here: in April–June many crop fields are still bare or newly planted (low NDVI), while
grassland is green and potholes hold snowmelt water.

The wetness coefficients were derived for Landsat 5 TM reflectance. We apply them to all sensors unchanged,
consistent with our decision not to adjust Landsat 8 (noted in the README limitations).

In [6]:
# Crist (1985) Tasseled Cap wetness coefficients for TM reflectance,
# in the order blue, green, red, nir, swir1, swir2
TC_WETNESS = [0.0315, 0.2021, 0.3102, 0.1594, -0.6806, -0.6109]


def add_indices(img):
    """Return the composite with NDVI, MNDWI and Tasseled Cap wetness bands added."""
    ndvi = img.normalizedDifference(["nir", "red"]).rename("ndvi")
    mndwi = img.normalizedDifference(["green", "swir1"]).rename("mndwi")
    # Multiply each band by its coefficient, then add them up across bands
    tcw = (
        img.select(BANDS)
        .multiply(ee.Image.constant(TC_WETNESS))
        .reduce(ee.Reducer.sum())
        .rename("tcw")
    )
    return img.addBands([ndvi, mndwi, tcw])


composites = {year: add_indices(img) for year, img in composites.items()}
print("Bands now in each composite:", composites[2020].bandNames().getInfo())

Bands now in each composite: ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'ndvi', 'mndwi', 'tcw']


## Step 2b — Sanity check: index summaries per snapshot

For each snapshot we print the 10th percentile, median and 90th percentile of each index across the watershed,
plus the share of pixels with **MNDWI > 0**. That last number is only a rough "looks like open water" indicator
(no classification yet), but it shows how wet each window was before we classify anything.

In [7]:
import pandas as pd

rows = []
for year, img in composites.items():
    pct = img.select(["ndvi", "mndwi", "tcw"]).reduceRegion(
        reducer=ee.Reducer.percentile([10, 50, 90]),
        geometry=aoi, scale=30, maxPixels=1e10,
    )
    # Mean of a 0/1 image = fraction of pixels where the condition is true
    water_like = img.select("mndwi").gt(0).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=30, maxPixels=1e10,
    )
    row = {"year": year, **pct.getInfo()}
    row["pct_pixels_mndwi_gt_0"] = water_like.get("mndwi").getInfo() * 100
    rows.append(row)

index_summary = pd.DataFrame(rows).set_index("year")
# Order columns as index_p10, index_p50, index_p90
cols = [f"{b}_p{p}" for b in ["ndvi", "mndwi", "tcw"] for p in (10, 50, 90)] + ["pct_pixels_mndwi_gt_0"]
index_summary = index_summary[cols].round(3)
index_summary

,ndvi_p10,ndvi_p50,ndvi_p90,mndwi_p10,mndwi_p50,mndwi_p90,tcw_p10,tcw_p50,tcw_p90,pct_pixels_mndwi_gt_0
year,,,,,,,,,,
2000,0.223,0.363,0.566,-0.551,-0.488,-0.356,-0.287,-0.201,-0.092,4.867
2010,0.184,0.293,0.535,-0.543,-0.504,-0.363,-0.319,-0.214,-0.089,5.059
2020,0.231,0.332,0.551,-0.566,-0.527,-0.434,-0.330,-0.252,-0.123,5.187


## Step 2c — Look at the indices on a map

Blue = wet / water, brown = dry, green = vegetated. Compare 2000 and 2020 for each index with the layer control.

In [8]:
m2 = geemap.Map(basemap="Esri.WorldImagery")
m2.centerObject(aoi, 10)

ndvi_vis = {"min": -0.2, "max": 0.6, "palette": ["8c510a", "f6e8c3", "1a9850"]}
mndwi_vis = {"min": -0.6, "max": 0.4, "palette": ["8c510a", "f5f5f5", "2166ac"]}
tcw_vis = {"min": -0.35, "max": 0.0, "palette": ["8c510a", "f5f5f5", "2166ac"]}

for year in (2000, 2020):
    img = composites[year]
    m2.addLayer(img.select("ndvi"), ndvi_vis, f"{year} NDVI", shown=False)
    m2.addLayer(img.select("tcw"), tcw_vis, f"{year} TC wetness", shown=False)
    m2.addLayer(img.select("mndwi"), mndwi_vis, f"{year} MNDWI", shown=(year == 2020))

m2.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": "yellow"}, "Pipestem boundary")
m2

## Step 3a — Reference data: NWI wetlands and the Cropland Data Layer

To train a classifier we need **labelled examples**: pixels where we already know the answer. Instead of
hand-digitizing, we borrow labels from two authoritative public datasets:

- **National Wetlands Inventory (NWI)**, USFWS. Wetland polygons for Pipestem were mapped from 2017–2019
  imagery. We downloaded the USFWS HUC8 file, clipped it to the watershed with GeoPandas
  (`scripts/prepare_nwi.py`), and uploaded it as an Earth Engine asset.
  Polygons coded `Pf` (*palustrine farmed*) are wetlands that are being cropped; we flag them with `farmed = 1`.
- **Cropland Data Layer (CDL)**, USDA NASS. A yearly 30 m crop-type map, available for this watershed every year
  since 1997. Its `cropland` band gives a crop-type code per pixel (e.g. 5 = soybeans, 61 = fallow/idle, 176 = grass/pasture).

**Defining "cultivated" the same way for every year.** Recent CDL years also have a ready-made `cultivated` band,
but older years (before 2008) don't. So we define cultivated ourselves from the crop codes (1–61, 66–77 and
204–254 are crops, including fallow/idle) and use that rule for all years. Below we check that our rule matches
the official `cultivated` band in 2020.

We also print the 2020 CDL class breakdown to see what the landscape looks like.

In [9]:
NWI_ASSET = "projects/agriculture-wetland-loss/assets/nwi_pipestem"
nwi = ee.FeatureCollection(NWI_ASSET)
print("NWI polygons:", nwi.size().getInfo(), "| farmed (Pf):", nwi.filter(ee.Filter.eq("farmed", 1)).size().getInfo())


def cdl_cropland(year):
    """The CDL crop-type band for one year."""
    return ee.Image(f"USDA/NASS/CDL/{year}").select("cropland")


def is_cultivated(crop):
    """1 where the CDL code is a crop (including fallow/idle, 61), 0 otherwise."""
    return (crop.gte(1).And(crop.lte(61))
            .Or(crop.gte(66).And(crop.lte(77)))
            .Or(crop.gte(204).And(crop.lte(254))))


# CDL water and wetland codes, old legend (83 water, 87 wetlands) and new legend (111, 190, 195)
WATER_WETLAND_CODES = [83, 87, 111, 190, 195]

cdl_proj = cdl_cropland(2020).projection()  # CDL's native 30 m Albers (EPSG:5070) grid; we sample on this grid

# Check: does our code-based rule match the official 2020 "cultivated" band (2 = cultivated)?
official = ee.Image("USDA/NASS/CDL/2020").select("cultivated").eq(2)
agreement = is_cultivated(cdl_cropland(2020)).eq(official).reduceRegion(
    ee.Reducer.mean(), aoi, crs=cdl_proj, scale=30, maxPixels=1e10).getInfo()
print(f"Our 'cultivated' rule agrees with the official 2020 cultivated band on {100 * list(agreement.values())[0]:.2f}% of pixels")

# 2020 CDL class breakdown inside the watershed
hist = cdl_cropland(2020).reduceRegion(
    ee.Reducer.frequencyHistogram(), aoi, scale=30, maxPixels=1e10
).get("cropland").getInfo()
# The class codes and names are stored as comma-separated strings in the image metadata
codes = ee.Image("USDA/NASS/CDL/2020").get("cropland_class_values").getInfo().split(",")
names = ee.Image("USDA/NASS/CDL/2020").get("cropland_class_names").getInfo().split(",")
assert len(codes) == len(names), "CDL code/name lists don't line up"
code_to_name = dict(zip(codes, names))

cdl_breakdown = (
    pd.DataFrame({"code": list(hist), "pixels": list(hist.values())})
    .assign(name=lambda d: d.code.map(code_to_name),
            percent=lambda d: (100 * d.pixels / d.pixels.sum()).round(1))
    .sort_values("pixels", ascending=False)
    .head(12)
    .set_index("code")
)
cdl_breakdown

NWI polygons: 44854 | farmed (Pf): 9191


Our 'cultivated' rule agrees with the official 2020 cultivated band on 91.80% of pixels


,pixels,name,percent
code,,,
176,746567.674510,Grass/Pasture,24.2
5,682063.427451,Soybeans,22.1
61,605600.266667,Fallow/Idle Cropland,19.7
195,231480.737255,Herbaceous Wetlands,7.5
111,221370.552941,Open Water,7.2
23,177753.388235,Spring Wheat,5.8
1,117718.709804,Corn,3.8
121,76274.082353,Developed/Open Space,2.5
37,60845.149020,Other Hay/Non Alfalfa,2.0


## Step 3b — Build "confident" label masks for each snapshot

Each snapshot gets its own labels, built from the CDL for the **three years of its window** (e.g. 1999–2001 for 2000)
plus NWI. Good training labels matter more than lots of them, so each class has strict rules. A pixel only becomes
a candidate if **all** of its rules hold:

| Class | Rule (CDL years = the snapshot's 3-year window) |
|---|---|
| **0 — Wetland/water** | Inside a non-farmed NWI polygon of the right type (below), **and** not cropped in any of the CDL years (fallow/idle allowed, see below) |
| **1 — Cropland** | Cultivated in all three CDL years, **not** fallow/idle (61) in any of them, and outside all NWI polygons |
| **2 — Grassland/other** | Not cultivated in any CDL year, not CDL water/wetland in any of them, and outside all NWI polygons |

**Two kinds of wetland.** Prairie potholes range from open water to marsh full of cattails and sedges.
If we only sampled "wetland" at random, most points would land in big lakes (the easiest pixels to keep),
and the model would learn *wetland = open water*. So the wetland class is built from two **strata**
(sub-groups), sampled separately and then merged into one class:
- **open water**: NWI `Lake` and `Freshwater Pond`
- **emergent marsh**: NWI `Freshwater Emergent Wetland`

Extra choices, all to avoid mislabelled pixels:
- **Three CDL years per snapshot** filters out one-year CDL errors and crop rotations into grass.
- **Smoothing the CDL first.** Older CDL years are "speckled": scattered single pixels with a different code
  inside a uniform field or marsh. Before applying the rules we pass a 3×3 **majority filter** over each CDL year
  (each pixel takes the most common code around it), which removes the speckle but keeps real field boundaries.
  Without it, almost no 2000 grassland or marsh pixels survived the rules (0.6 km² of marsh).
- **Fallow doesn't disqualify a wetland.** Older CDL often codes marsh as "fallow/idle" (61), so for the wetland
  class we only exclude pixels that were actually *cropped*. (Farmed NWI wetlands, `Pf`, are already excluded.)
- **"Pure pixel" rule:** a 30 m pixel on a field or wetland edge is a mix of both. We keep a pixel only if
  the whole 3×3 block around it (90 m × 90 m) belongs to the same stratum. Trade-off: this drops the tiniest
  potholes from training, so the models learn wetlands mostly from medium and large ones.

**Assumption to keep in mind:** NWI was mapped from 2017–2019 imagery. Using it for 2000 and 2010 assumes those
wetlands already existed then (and the "not cultivated in that window" rule removes ones that were being farmed).
A wetland that only formed after 2000 would be mislabelled in the 2000 training data.

In [10]:
# NWI as 0/1 rasters (Pf polygons have wet_type "Other", so they're excluded from both wetland strata)
nwi_any = ee.Image(0).byte().paint(nwi, 1)
nwi_open = ee.Image(0).byte().paint(nwi.filter(ee.Filter.inList("wet_type", ["Lake", "Freshwater Pond"])), 1)
nwi_emergent = ee.Image(0).byte().paint(nwi.filter(ee.Filter.eq("wet_type", "Freshwater Emergent Wetland")), 1)

# Final classes the models will learn
CLASS_NAMES = {0: "wetland/water", 1: "cropland", 2: "grassland/other"}
# One colour per class, used in every map and chart (blue, orange, aqua: checked to stay
# distinguishable for the common types of colour blindness)
CLASS_PALETTE = ["2a78d6", "eb6834", "1baf7a"]
# Sampling strata: two wetland sub-groups that both become class 0
STRATUM_NAMES = {0: "wetland: open water", 1: "cropland", 2: "grassland/other", 3: "wetland: emergent marsh"}
STRATUM_TO_CLASS = [0, 1, 2, 0]


def pure(mask):
    """Keep a pixel only if its whole 3x3 neighbourhood is in the mask (computed on the 30 m CDL grid)."""
    return mask.reproject(cdl_proj).focalMin(radius=1, kernelType="square", units="pixels")


def smoothed_cdl(year):
    """CDL crop codes after a 3x3 majority filter, which removes isolated 'speckle' pixels."""
    return cdl_cropland(year).reproject(cdl_proj).focalMode(radius=1, kernelType="square", units="pixels")


def label_strata(year):
    """One-band image of training strata (0-3) for a snapshot, using smoothed CDL for year-1..year+1 and NWI."""
    crops = ee.ImageCollection([smoothed_cdl(y) for y in (year - 1, year, year + 1)])
    cultivated = crops.map(is_cultivated)
    cropped = crops.map(lambda im: is_cultivated(im).And(im.neq(61)))  # cultivated, but fallow doesn't count

    always_cultivated = cultivated.min()       # 1 only if cultivated every year
    never_cultivated = cultivated.max().Not()  # 1 only if cultivated in no year
    never_cropped = cropped.max().Not()        # 1 only if never cropped (fallow/idle allowed)
    ever_fallow = crops.map(lambda im: im.eq(61)).max()
    ever_water_wetland = crops.map(
        lambda im: im.remap(WATER_WETLAND_CODES, [1] * len(WATER_WETLAND_CODES), 0)).max()

    open_water = pure(nwi_open.And(never_cropped))
    emergent = pure(nwi_emergent.And(never_cropped))
    cropland = pure(always_cultivated.And(ever_fallow.Not()).And(nwi_any.Not()))
    grass = pure(never_cultivated.And(ever_water_wetland.Not()).And(nwi_any.Not()))

    return (
        ee.Image(0).where(cropland, 1).where(grass, 2).where(emergent, 3)
        .updateMask(open_water.Or(cropland).Or(grass).Or(emergent))
        .rename("stratum").reproject(cdl_proj).clip(aoi)
    )


strata = {year: label_strata(year) for year in SNAPSHOTS}

# How much candidate area does each stratum have in each snapshot? (km², 30 m pixels = 900 m²)
candidates = {}
for year, img in strata.items():
    h = img.reduceRegion(ee.Reducer.frequencyHistogram(), aoi, crs=cdl_proj, scale=30,
                         maxPixels=1e10).get("stratum").getInfo()
    candidates[year] = {STRATUM_NAMES[int(k)]: v * 900 / 1e6 for k, v in h.items()}
print("Candidate area per stratum (km²):")
pd.DataFrame(candidates).loc[list(STRATUM_NAMES.values())].round(1)

Candidate area per stratum (km²):


,2000,2010,2020
wetland: open water,96.1,99.9,100.4
cropland,442.3,751.6,742.3
grassland/other,179.3,493.3,388.5
wetland: emergent marsh,7.9,9.5,11.1


## Step 3c — Sample training points and attach each snapshot's Landsat values

For each snapshot, `stratifiedSample` draws a set number of random points from each stratum: **150 open water +
150 emergent marsh (= 300 wetland), 300 cropland, 300 grassland/other**. Equal class sizes keep a model from being
swamped by the most common class. The fixed `seed` makes the draw repeatable. Each point then gets its final
`class` (both wetland strata → class 0) while keeping its `stratum` for checking.

Then `sampleRegions` reads **that snapshot's** composite under each point: the six reflectance bands plus NDVI,
MNDWI and TC wetness. These nine numbers are the **features** the random forest learns from. Points on a pixel with
no clear Landsat data are dropped automatically, so we print the final counts.

In [11]:
FEATURES = BANDS + ["ndvi", "mndwi", "tcw"]
STRATUM_POINTS = {0: 150, 1: 300, 2: 300, 3: 150}


def sample_training(year):
    """Stratified random points for one snapshot, with that snapshot's nine Landsat features attached."""
    points = strata[year].stratifiedSample(
        numPoints=0,  # ignored: classPoints below sets the count per stratum
        classBand="stratum", classValues=list(STRATUM_POINTS), classPoints=list(STRATUM_POINTS.values()),
        region=aoi, projection=cdl_proj, scale=30, seed=42, geometries=True,
    )
    # Map each stratum to its final class (0 open water -> 0, 3 emergent -> 0)
    points = points.map(lambda f: f.set("class", ee.List(STRATUM_TO_CLASS).get(f.get("stratum"))))
    return composites[year].select(FEATURES).sampleRegions(
        collection=points, properties=["class", "stratum"], projection=cdl_proj, scale=30, geometries=True,
    )


training, training_gdf = {}, {}
for year in SNAPSHOTS:
    training[year] = sample_training(year)
    # Bring the samples to the client as a GeoDataFrame (~900 rows) and save them for reproducibility
    gdf = geemap.ee_to_gdf(training[year])
    gdf["class_name"] = gdf["class"].map(CLASS_NAMES)
    gdf["stratum_name"] = gdf["stratum"].map(STRATUM_NAMES)
    gdf.to_file(f"outputs/training_points_{year}.geojson", driver="GeoJSON")
    training_gdf[year] = gdf

print("Training points per stratum:")
print(pd.DataFrame({y: g["stratum_name"].value_counts() for y, g in training_gdf.items()}).to_string(), "\n")

# Sanity check: average values per stratum and snapshot should look physically sensible
pd.concat({y: g.groupby("stratum_name")[["ndvi", "mndwi", "tcw", "nir", "swir1"]].mean()
           for y, g in training_gdf.items()}, names=["snapshot"]).round(3)

Training points per stratum:
                         2000  2010  2020
stratum_name                             
cropland                  300   300   300
grassland/other           300   300   300
wetland: open water       150   150   150
wetland: emergent marsh   150   150   150 



ndvi  mndwi    tcw    nir  swir1
snapshot stratum_name                                              
2000     cropland                 0.296 -0.522 -0.265  0.198  0.292
         grassland/other          0.516 -0.483 -0.178  0.299  0.271
         wetland: emergent marsh  0.339 -0.230 -0.074  0.155  0.128
         wetland: open water      0.005  0.204 -0.006  0.051  0.037
2010     cropland                 0.231 -0.526 -0.297  0.203  0.332
         grassland/other          0.477 -0.501 -0.189  0.269  0.272
         wetland: emergent marsh  0.275 -0.242 -0.077  0.133  0.125
         wetland: open water     -0.051  0.315  0.002  0.042  0.027
2020     cropland                 0.297 -0.527 -0.297  0.241  0.341
         grassland/other          0.529 -0.549 -0.206  0.289  0.285
         wetland: emergent marsh  0.404 -0.440 -0.152  0.226  0.220
         wetland: open water     -0.240  0.425  0.003  0.020  0.017

## Step 3d — Check the labels against NAIP aerial photos

Automated labels can still be wrong (e.g., a dry pothole that NWI calls wetland, or a CDL pixel in a farmyard),
so we **look** at a sample. NAIP is 0.6–1 m aerial photography. It's available here for 2020 and 2010 (but not for
2000; the earliest NAIP for this watershed is 2003), so we check those two snapshots.
Note that NAIP is flown in **summer**, while our Landsat composites are **spring**, so ponds may look smaller
and crops greener in NAIP than in Landsat.

1. **Static chip grids** (saved to `outputs/figures/training_points_naip_check_<year>.png`): 5 random points per
   stratum, each shown as a 240 m × 240 m NAIP chip with the point at the red cross.
2. **Interactive map**: all 2020 training points over NAIP 2020 (blue = wetland/water, orange = cropland,
   aqua = grassland/other). Zoom in anywhere to spot-check.

In [12]:
import io
import urllib.request

import matplotlib.pyplot as plt
from PIL import Image


def naip_mosaic(year):
    return (ee.ImageCollection("USDA/NAIP/DOQQ").filterBounds(aoi)
            .filterDate(f"{year}-01-01", f"{year + 1}-01-01").mosaic())


def naip_chip_grid(year, n_per_stratum=5):
    """Plot NAIP chips around random training points of one snapshot and save the figure."""
    naip = naip_mosaic(year)
    check = training_gdf[year].groupby("stratum").sample(n_per_stratum, random_state=1)
    fig, axes = plt.subplots(4, n_per_stratum, figsize=(2.6 * n_per_stratum, 10.8))
    for row, stratum in enumerate([0, 3, 1, 2]):  # both wetland rows first
        group = check[check["stratum"] == stratum]
        for col, (_, pt) in enumerate(group.iterrows()):
            box = ee.Geometry.Point([pt.geometry.x, pt.geometry.y]).buffer(120).bounds()
            url = naip.getThumbURL({"bands": ["R", "G", "B"], "min": 0, "max": 255,
                                    "region": box, "dimensions": 256, "format": "png"})
            chip = Image.open(io.BytesIO(urllib.request.urlopen(url).read()))
            ax = axes[row, col]
            ax.imshow(chip)
            ax.plot(chip.width / 2, chip.height / 2, "r+", markersize=14, markeredgewidth=2)
            ax.set_xticks([]); ax.set_yticks([])
            if col == 0:
                ax.set_ylabel(STRATUM_NAMES[stratum].replace(": ", ":\n"), fontsize=11)
    fig.suptitle(f"{year} training point check: NAIP {year}, 240 m chips, point at red cross", fontsize=12)
    fig.tight_layout()
    fig.savefig(f"outputs/figures/training_points_naip_check_{year}.png", dpi=150)
    plt.show()


for year in (2020, 2010):
    naip_chip_grid(year)

In [13]:
m3 = geemap.Map(basemap="Esri.WorldImagery")
m3.centerObject(aoi, 10)
m3.addLayer(naip_mosaic(2020), {"bands": ["R", "G", "B"], "min": 0, "max": 255}, "NAIP 2020")
m3.addLayer(nwi.style(color="00ffff", fillColor="00000000", width=1), {}, "NWI outlines", shown=False)

for cls, color in enumerate(CLASS_PALETTE):
    pts = training[2020].filter(ee.Filter.eq("class", cls))
    m3.addLayer(pts.style(color=color, pointSize=4, fillColor=color), {}, f"2020 {CLASS_NAMES[cls]} points")

m3.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": "yellow"}, "Pipestem boundary")
m3

## Step 4a — Split each snapshot's points: 70% to train, 30% to test

We hide 30% of the points from each model while it learns, then use them to check how well it does on points
it has never seen. `randomColumn` gives every point a random number between 0 and 1; points below 0.7 go to
training and the rest to testing. The seed makes the split repeatable.

**Caveat:** this is a *random* split, so some test points sit close to training points (e.g. in the same lake).
Nearby pixels look alike, which makes the test a bit easier than classifying a brand-new area. The accuracies
below are therefore probably somewhat **optimistic**.

In [14]:
train_sets, test_sets = {}, {}
split_counts = {}
for year in SNAPSHOTS:
    split = training[year].randomColumn("random", seed=7)
    train_sets[year] = split.filter(ee.Filter.lt("random", 0.7))
    test_sets[year] = split.filter(ee.Filter.gte("random", 0.7))
    for name, fc in (("train", train_sets[year]), ("test", test_sets[year])):
        h = fc.aggregate_histogram("class").getInfo()
        split_counts[(year, name)] = {CLASS_NAMES[int(k)]: v for k, v in h.items()}

split_counts = pd.DataFrame(split_counts)
split_counts.loc["total"] = split_counts.sum()
split_counts

2000       2010       2020     
                train test train test train test
wetland/water     207   93   206   94   222   78
cropland          221   79   205   95   201   99
grassland/other   203   97   203   97   208   92
total             631  269   614  286   631  269

## Step 4b — Train one random forest per snapshot

A **random forest** is a collection of decision trees. Each tree is trained on a random subset of the points and
features and learns a series of yes/no questions (e.g. *"Is MNDWI above 0.1?"*). To classify a pixel, every
tree votes and the majority wins. It's a popular choice for land cover because it handles many features,
needs little tuning, and is hard to overfit badly.

We train **one model per snapshot**, each on its own year's points and imagery, so each model learns what the
classes looked like in that period (sensor, weather and farming practices all differ between 2000 and 2020).
Each uses 200 trees and Earth Engine's defaults for everything else. The table shows which features each model
relied on most.

In [15]:
models = {}
importance = {}
for year in SNAPSHOTS:
    models[year] = ee.Classifier.smileRandomForest(numberOfTrees=200, seed=42).train(
        features=train_sets[year], classProperty="class", inputProperties=FEATURES,
    )
    imp = pd.Series(models[year].explain().get("importance").getInfo())
    importance[year] = 100 * imp / imp.sum()

# Which features did each forest rely on most? (% of total importance; higher = more useful)
pd.DataFrame(importance).round(1).sort_values(2020, ascending=False)

,2000,2010,2020
swir2,14.2,15.2,14.2
nir,15.0,13.8,13.1
ndvi,10.4,11.1,12.7
mndwi,9.8,10.7,11.1
tcw,12.8,12.2,11.0
red,8.5,8.1,10.4
swir1,12.2,12.4,9.6
blue,8.5,8.1,9.1
green,8.5,8.5,8.7


## Step 4c — Test on the held-out 30%: confusion matrices and accuracy

The **confusion matrix** compares the true class (rows) with the model's prediction (columns). Numbers on the
diagonal are correct; everything off the diagonal is a mistake, and *where* it sits tells you which classes get mixed up.

- **Overall accuracy**: share of all test points classified correctly.
- **Producer's accuracy** (per class, read across a row): of the points that really are wetland, what share did
  the model find? Low = the class gets *missed* (omission error).
- **User's accuracy** (per class, read down a column): of the points the model *called* wetland, what share
  really are? Low = the class gets *over-predicted* (commission error).
- **Kappa**: agreement corrected for chance (1 = perfect, 0 = no better than random).

In [16]:
labels_order = [CLASS_NAMES[i] for i in range(3)]
tested = {}
summary_rows = []
for year in SNAPSHOTS:
    tested[year] = test_sets[year].classify(models[year])
    em = tested[year].errorMatrix("class", "classification")

    cm = pd.DataFrame(em.getInfo(), index=labels_order, columns=labels_order)
    cm.index.name, cm.columns.name = "true class", "predicted class"
    oa, kappa = em.accuracy().getInfo(), em.kappa().getInfo()
    producers = [row[0] for row in em.producersAccuracy().getInfo()]
    users = em.consumersAccuracy().getInfo()[0]

    cm.to_csv(f"outputs/confusion_matrix_{year}.csv")
    row = {"snapshot": year, "test_points": int(cm.values.sum()),
           "overall_accuracy": oa, "kappa": kappa}
    for i, name in enumerate(labels_order):
        row[f"producers_{name}"] = producers[i]
        row[f"users_{name}"] = users[i]
    summary_rows.append(row)

    print(f"===== {year}: overall accuracy {oa:.3f}, kappa {kappa:.3f} ({int(cm.values.sum())} test points)")
    print(cm.to_string(), "\n")

accuracy = pd.DataFrame(summary_rows).set_index("snapshot").round(3)
accuracy.to_csv("outputs/accuracy_by_snapshot.csv")
accuracy.T

===== 2000: overall accuracy 0.955, kappa 0.933 (269 test points)
predicted class  wetland/water  cropland  grassland/other
true class                                               
wetland/water               84         2                7
cropland                     1        77                1
grassland/other              1         0               96 



===== 2010: overall accuracy 0.986, kappa 0.979 (286 test points)
predicted class  wetland/water  cropland  grassland/other
true class                                               
wetland/water               92         0                2
cropland                     1        94                0
grassland/other              0         1               96 



===== 2020: overall accuracy 0.933, kappa 0.899 (269 test points)
predicted class  wetland/water  cropland  grassland/other
true class                                               
wetland/water               69         0                9
cropland                     1        92                6
grassland/other              2         0               90 



snapshot,2000,2010,2020
test_points,269.000,286.000,269.000
overall_accuracy,0.955,0.986,0.933
kappa,0.933,0.979,0.899
producers_wetland/water,0.903,0.979,0.885
users_wetland/water,0.977,0.989,0.958
producers_cropland,0.975,0.989,0.929
users_cropland,0.975,0.989,1.000
producers_grassland/other,0.990,0.990,0.978
users_grassland/other,0.923,0.980,0.857


## Step 4d — Which wetland type do the models miss?

The wetland class mixes open water and emergent marsh. Splitting each test result by **stratum** shows whether
a model handles both, or only the easy open water. Values are the share of test points classified correctly.

In [17]:
by_stratum = {}
for year in SNAPSHOTS:
    g = geemap.ee_to_gdf(tested[year])
    g["stratum_name"] = g["stratum"].map(STRATUM_NAMES)
    g["correct"] = g["classification"] == g["class"]
    by_stratum[(year, "n_test")] = g.groupby("stratum_name").size()
    by_stratum[(year, "share_correct")] = g.groupby("stratum_name")["correct"].mean().round(3)
pd.DataFrame(by_stratum)

2000                 2010                 2020  \
                        n_test share_correct n_test share_correct n_test   
stratum_name                                                               
cropland                    79         0.975     95         0.989     99   
grassland/other             97         0.990     97         0.990     92   
wetland: emergent marsh     51         0.824     44         0.955     40   
wetland: open water         42         1.000     50         1.000     38   

                                       
                        share_correct  
stratum_name                           
cropland                        0.929  
grassland/other                 0.978  
wetland: emergent marsh         0.775  
wetland: open water             1.000